# 01. NBA News Dataset Collection

Pipeline for collecting, filtering, and aggregating NBA news from GDELT GKG 2.1.

## 1. Research Goal

Objective: build a reproducible dataset of NBA news features for 2020-01-01 to 2022-12-31.

Hypothesis H4: team news background provides an additional but limited predictive signal for NBA game outcome forecasting.

## 2. Environment Setup

In [ ]:
import io
import re
import time
import random
import zipfile
import warnings
from pathlib import Path
from datetime import datetime, timedelta
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

## 3. Project Paths and Configuration

In [ ]:
# Пути проекта
PROJECT_ROOT = Path.cwd()
PATHS = {
    "raw": PROJECT_ROOT / "data/news/raw",
    "processed": PROJECT_ROOT / "data/news/processed",
    "reports": PROJECT_ROOT / "data/news/reports",
    "artifacts": PROJECT_ROOT / "artifacts/news",
    "docs": PROJECT_ROOT / "docs",
}
for p in PATHS.values():
    p.mkdir(parents=True, exist_ok=True)

# Параметры периода
TEST_START = "2020-01-01"
TEST_END = "2020-01-31"
FULL_START = "2020-01-01"
FULL_END = "2022-12-31"
GKG_FREQ_HOURS = 6
RANDOM_STATE = 42
GDELT_BASE_URL = "http://data.gdeltproject.org/gdeltv2"
REQUEST_TIMEOUT = 45
MAX_RETRIES = 2

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print("PROJECT_ROOT:", PROJECT_ROOT)
for k, v in PATHS.items():
    print(f"{k}: {v}")

PROJECT_ROOT: /content
raw: /content/data/news/raw
processed: /content/data/news/processed
reports: /content/data/news/reports
artifacts: /content/artifacts/news
docs: /content/docs


## 4. NBA Teams and Aliases

In [ ]:
# Список команд NBA
TEAM_ALIASES = {
    "ATL": ["atlanta hawks", "hawks"],
    "BOS": ["boston celtics", "celtics"],
    "BKN": ["brooklyn nets", "nets"],
    "CHA": ["charlotte hornets", "hornets"],
    "CHI": ["chicago bulls", "bulls"],
    "CLE": ["cleveland cavaliers", "cavaliers", "cavs"],
    "DAL": ["dallas mavericks", "mavericks", "mavs"],
    "DEN": ["denver nuggets", "nuggets"],
    "DET": ["detroit pistons", "pistons"],
    "GSW": ["golden state warriors", "warriors"],
    "HOU": ["houston rockets", "rockets"],
    "IND": ["indiana pacers", "pacers"],
    "LAC": ["los angeles clippers", "la clippers", "clippers"],
    "LAL": ["los angeles lakers", "la lakers", "lakers"],
    "MEM": ["memphis grizzlies", "grizzlies"],
    "MIA": ["miami heat", "heat"],
    "MIL": ["milwaukee bucks", "bucks"],
    "MIN": ["minnesota timberwolves", "timberwolves", "wolves"],
    "NOP": ["new orleans pelicans", "pelicans"],
    "NYK": ["new york knicks", "knicks"],
    "OKC": ["oklahoma city thunder", "thunder"],
    "ORL": ["orlando magic", "magic"],
    "PHI": ["philadelphia 76ers", "76ers", "sixers"],
    "PHX": ["phoenix suns", "suns"],
    "POR": ["portland trail blazers", "trail blazers", "blazers"],
    "SAC": ["sacramento kings", "kings"],
    "SAS": ["san antonio spurs", "spurs"],
    "TOR": ["toronto raptors", "raptors"],
    "UTA": ["utah jazz", "jazz"],
    "WAS": ["washington wizards", "wizards"],
}
AMBIGUOUS_ALIASES = {"heat", "magic", "suns", "kings", "rockets", "warriors"}
display(pd.DataFrame({"team": TEAM_ALIASES.keys(), "aliases": [", ".join(v) for v in TEAM_ALIASES.values()]}))

,team,aliases
0,ATL,"atlanta hawks, hawks"
1,BOS,"boston celtics, celtics"
2,BKN,"brooklyn nets, nets"
3,CHA,"charlotte hornets, hornets"
4,CHI,"chicago bulls, bulls"
5,CLE,"cleveland cavaliers, cavaliers, cavs"
6,DAL,"dallas mavericks, mavericks, mavs"
7,DEN,"denver nuggets, nuggets"
8,DET,"detroit pistons, pistons"
9,GSW,"golden state warriors, warriors"


## 5. Text Normalization and Team Detection

In [ ]:
# Нормализация текста
def normalize_text(text: str) -> str:
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return ""
    t = str(text).lower()
    t = re.sub(r"<[^>]+>", " ", t)
    t = re.sub(r"[^a-z0-9\s\-']", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

def _contains_phrase(text: str, phrase: str) -> bool:
    return re.search(rf"\b{re.escape(phrase)}\b", text) is not None

def detect_teams(title: str) -> dict:
    norm = normalize_text(title)
    matched_teams, matched_aliases = [], []
    for team, aliases in TEAM_ALIASES.items():
        for a in aliases:
            if _contains_phrase(norm, a):
                matched_teams.append(team)
                matched_aliases.append(a)
                break
    return {
        "matched_teams": sorted(set(matched_teams)),
        "matched_aliases": sorted(set(matched_aliases)),
        "has_team_alias": bool(matched_teams),
    }

## 6. NBA Relevance Filtering

In [ ]:
# Фильтрация релевантности
NBA_CONTEXT_KEYWORDS = {
    "nba", "basketball", "playoffs", "regular season", "matchup", "roster", "coach", "player",
    "trade", "injury", "draft", "finals", "eastern conference", "western conference", "all-star",
    "game", "win", "loss", "season", "team", "lineup",
}

SPORTS_SOURCES_WHITELIST = {
    "nba.com", "espn.com", "cbssports.com", "bleacherreport.com", "si.com",
    "sports.yahoo.com", "clutchpoints.com", "theathletic.com", "foxsports.com", "nbcsports.com",
}

EXCLUSION_PATTERNS = [
    r"\bnfl\b", r"\bmlb\b", r"\bnhl\b", r"\bsoccer\b", r"\bcricket\b", r"\brugby\b",
    r"college football", r"\bpolitics\b", r"\bwar\b", r"\bmilitary\b", r"missile",
    r"\bweather\b", r"heat wave", r"\bfire\b", r"\bfinance\b", r"\bentertainment\b",
]

def is_sports_source(source_name: str, url: str) -> bool:
    s = normalize_text(source_name)
    host = urlparse(url).netloc.lower() if isinstance(url, str) else ""
    cands = {s, host}
    if host:
        cands.add(".".join(host.split(".")[-2:]))
    return any(any(w in c for w in SPORTS_SOURCES_WHITELIST) for c in cands)

def has_nba_context(text: str) -> bool:
    t = normalize_text(text)
    return any(_contains_phrase(t, kw) for kw in NBA_CONTEXT_KEYWORDS)

def has_exclusion_context(text: str) -> bool:
    t = normalize_text(text)
    return any(re.search(p, t) for p in EXCLUSION_PATTERNS)

def is_nba_relevant(title: str, source_name: str, url: str) -> dict:
    team_info = detect_teams(title)
    nba_ctx = has_nba_context(title)
    sports_src = is_sports_source(source_name, url)
    excl = has_exclusion_context(title)
    ambiguous_hit = any(a in AMBIGUOUS_ALIASES for a in team_info["matched_aliases"])
    ambiguous_requires_context = ambiguous_hit and not (nba_ctx or sports_src)

    is_relevant = (
        team_info["has_team_alias"]
        and (nba_ctx or sports_src)
        and not excl
        and not ambiguous_requires_context
    )

    if not team_info["has_team_alias"]:
        reason = "filtered:no_team_alias"
    elif excl and not (nba_ctx or sports_src):
        reason = "filtered:exclusion_context"
    elif ambiguous_requires_context:
        reason = "filtered:ambiguous_alias_without_nba_context"
    elif not (nba_ctx or sports_src):
        reason = "filtered:no_nba_context_or_sports_source"
    else:
        reason = "kept:team_alias+context"

    return {
        "is_relevant": is_relevant,
        "matched_teams": team_info["matched_teams"],
        "has_team_alias": team_info["has_team_alias"],
        "has_nba_context": nba_ctx,
        "is_sports_source": sports_src,
        "has_exclusion_context": excl,
        "relevance_reason": reason,
    }

## 7. Keyword Sentiment and NLP Flags

In [ ]:
# Расчёт тональности
POSITIVE_KEYWORDS = [
    "win", "wins", "won", "victory", "beat", "dominate", "comeback", "strong", "improve",
    "career-high", "record", "milestone", "healthy", "return", "cleared", "playoff berth",
    "clinched", "advance", "championship",
]
NEGATIVE_KEYWORDS = [
    "lose", "loss", "lost", "injury", "injured", "hurt", "ruled out", "struggle", "collapse",
    "suspended", "ejected", "worst", "poor", "waived", "released", "eliminated", "trade request",
]

def _count_hits(text: str, keywords: list[str]) -> int:
    t = normalize_text(text)
    return sum(int(_contains_phrase(t, k)) for k in keywords)

def keyword_sentiment(text: str) -> dict:
    p = _count_hits(text, POSITIVE_KEYWORDS)
    n = _count_hits(text, NEGATIVE_KEYWORDS)
    score = p - n
    label = "positive" if score > 0 else "negative" if score < 0 else "neutral"
    return {
        "sentiment_label_kw": label,
        "sentiment_score_kw": int(score),
        "positive_hits": int(p),
        "negative_hits": int(n),
    }

def build_keyword_flags(text: str) -> dict:
    t = normalize_text(text)
    return {
        "has_injury_kw": int(any(_contains_phrase(t, k) for k in ["injury", "injured", "ruled out", "hurt"])),
        "has_trade_kw": int(any(_contains_phrase(t, k) for k in ["trade", "trade request", "waived", "released"])),
        "has_win_kw": int(any(_contains_phrase(t, k) for k in ["win", "wins", "won", "victory", "beat"])),
        "has_loss_kw": int(any(_contains_phrase(t, k) for k in ["lose", "loss", "lost", "eliminated"])),
        "has_playoff_kw": int(any(_contains_phrase(t, k) for k in ["playoff", "playoffs", "finals", "clinched"])),
        "has_roster_kw": int(any(_contains_phrase(t, k) for k in ["roster", "lineup", "coach", "player"])),
    }

## 8. GDELT GKG Processing

In [ ]:
# Обработка GDELT
GKG_COLUMNS = [
    "GKGRECORDID", "DATE", "SourceCollectionIdentifier", "SourceCommonName", "DocumentIdentifier",
    "Counts", "V2Counts", "Themes", "V2Themes", "Locations", "V2Locations", "Persons", "V2Persons",
    "Organizations", "V2Organizations", "V2Tone", "Dates", "GCAM", "SharingImage", "RelatedImages",
    "SocialImageEmbeds", "SocialVideoEmbeds", "Quotations", "AllNames", "Amounts", "TranslationInfo", "Extras",
]

def generate_gkg_urls(start_date: str, end_date: str, freq_hours: int = 6) -> list[str]:
    start_dt = datetime.strptime(start_date, "%Y-%m-%d")
    end_dt = datetime.strptime(end_date, "%Y-%m-%d")
    urls = []
    cur = start_dt
    while cur <= end_dt:
        for h in range(0, 24, freq_hours):
            ts = cur.replace(hour=h, minute=0, second=0).strftime("%Y%m%d%H%M%S")
            urls.append(f"{GDELT_BASE_URL}/{ts}.gkg.csv.zip")
        cur += timedelta(days=1)
    return urls

def extract_page_title(extras: str) -> str:
    if not isinstance(extras, str):
        return ""
    m = re.search(r"<PAGE_TITLE>(.*?)</PAGE_TITLE>", extras, flags=re.IGNORECASE | re.DOTALL)
    return m.group(1).strip() if m else ""

def parse_gdelt_tone(v2tone: str) -> float:
    if not isinstance(v2tone, str) or not v2tone:
        return np.nan
    try:
        return float(v2tone.split(",")[0])
    except Exception:
        return np.nan

def process_gkg_file(url: str) -> pd.DataFrame:
    for attempt in range(MAX_RETRIES + 1):
        try:
            r = requests.get(url, timeout=REQUEST_TIMEOUT)
            if r.status_code != 200:
                return pd.DataFrame()
            with zipfile.ZipFile(io.BytesIO(r.content)) as zf:
                fname = zf.namelist()[0]
                with zf.open(fname) as f:
                    df = pd.read_csv(f, sep="\t", header=None, names=GKG_COLUMNS, dtype=str, on_bad_lines="skip", low_memory=False)
            if df.empty:
                return df
            df = df[["DATE", "SourceCommonName", "DocumentIdentifier", "V2Tone", "Extras"]].copy()
            df["datetime"] = pd.to_datetime(df["DATE"], format="%Y%m%d%H%M%S", errors="coerce")
            df = df[df["datetime"].notna()].copy()
            df["news_date"] = df["datetime"].dt.date
            df["title"] = df["Extras"].apply(extract_page_title)
            df["url"] = df["DocumentIdentifier"].fillna("").astype(str)
            df["source_name"] = df["SourceCommonName"].fillna("").astype(str)
            df["gdelt_tone"] = df["V2Tone"].apply(parse_gdelt_tone)
            df = df[["news_date", "datetime", "title", "url", "source_name", "gdelt_tone"]]
            return df[df["title"].str.len() > 0].copy()
        except Exception:
            if attempt == MAX_RETRIES:
                return pd.DataFrame()
            time.sleep(0.5)
    return pd.DataFrame()

def process_month(start_date: str, end_date: str, save_prefix: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    urls = generate_gkg_urls(start_date, end_date, GKG_FREQ_HOURS)
    chunks = []
    for url in tqdm(urls, desc=f"GKG {save_prefix}"):
        part = process_gkg_file(url)
        if not part.empty:
            chunks.append(part)

    base = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame(columns=["news_date", "datetime", "title", "url", "source_name", "gdelt_tone"])

    rows = []
    for row in base.itertuples(index=False):
        rel = is_nba_relevant(row.title, row.source_name, row.url)
        sent = keyword_sentiment(row.title)
        flags = build_keyword_flags(row.title)
        teams = rel["matched_teams"] if rel["matched_teams"] else [None]
        for team in teams:
            rows.append({
                "news_date": row.news_date,
                "datetime": row.datetime,
                "team": team,
                "matched_teams": ",".join(rel["matched_teams"]),
                "title": row.title,
                "url": row.url,
                "source_name": row.source_name,
                "gdelt_tone": row.gdelt_tone,
                **sent,
                **flags,
                "title_len": len(str(row.title)),
                "title_word_count": len(normalize_text(row.title).split()),
                "has_nba_context": rel["has_nba_context"],
                "is_sports_source": rel["is_sports_source"],
                "has_exclusion_context": rel["has_exclusion_context"],
                "relevance_decision": rel["is_relevant"],
                "relevance_reason": rel["relevance_reason"],
            })

    raw = pd.DataFrame(rows)
    clean = raw[(raw["relevance_decision"]) & (raw["team"].notna())].copy() if not raw.empty else raw.copy()

    raw.to_parquet(PATHS["raw"] / f"{save_prefix}_raw.parquet", index=False)
    clean.to_parquet(PATHS["processed"] / f"{save_prefix}_clean.parquet", index=False)
    return raw, clean

def load_all_months(prefixes: list[str], folder: Path) -> pd.DataFrame:
    files = [folder / f"{p}.parquet" for p in prefixes]
    existing = [f for f in files if f.exists()]
    if not existing:
        return pd.DataFrame()
    return pd.concat([pd.read_parquet(f) for f in existing], ignore_index=True)

## 9. Test Run on One Month

In [ ]:
test_prefix = "nba_news_2020_01"
test_raw, test_clean = process_month(TEST_START, TEST_END, test_prefix)

print("raw size:", len(test_raw))
print("clean size:", len(test_clean))
display(test_clean["team"].value_counts().head(15).rename("top_teams"))
display(test_clean["source_name"].value_counts().head(15).rename("top_sources"))
display(test_clean[["news_date", "team", "title", "source_name", "relevance_reason"]].head(10))
display(test_raw[~test_raw["relevance_decision"]][["news_date", "team", "title", "source_name", "relevance_reason"]].head(10))

amb = test_raw[test_raw["title"].str.lower().str.contains(r"\\b(heat|magic|suns|kings|rockets|warriors)\\b", regex=True, na=False)]
display(amb[["news_date", "team", "title", "source_name", "relevance_decision", "relevance_reason"]].head(30))

GKG nba_news_2020_01:   0%|          | 0/124 [00:00<?, ?it/s]

raw size: 204543
clean size: 578


,top_teams
team,
LAL,81
LAC,39
MIL,33
PHI,27
NOP,24
NYK,24
GSW,23
BOS,22
CLE,22


,top_sources
source_name,
iheart.com,52
clutchpoints.com,32
kesn1033.com,18
ktop1490.com,14
bleacherreport.com,14
fantasypros.com,14
espn.com,13
lakersnation.com,13
ibtimes.com,13


,news_date,team,title,source_name,relevance_reason
545,2020-01-01,IND,Pacers hand Embiid-less 76ers third straight l...,740thefan.com,kept:team_alias+context
546,2020-01-01,PHI,Pacers hand Embiid-less 76ers third straight l...,740thefan.com,kept:team_alias+context
1183,2020-01-01,BOS,"Celtics bounce back, hand Hornets sixth straig...",740thefan.com,kept:team_alias+context
1184,2020-01-01,CHA,"Celtics bounce back, hand Hornets sixth straig...",740thefan.com,kept:team_alias+context
1313,2020-01-01,MIL,NBA-best Bucks take on struggling Timberwolves,kesn1033.com,kept:team_alias+context
1314,2020-01-01,MIN,NBA-best Bucks take on struggling Timberwolves,kesn1033.com,kept:team_alias+context
1339,2020-01-01,GSW,"Warriors look like a new team in thrilling, un...",knbr.com,kept:team_alias+context
1340,2020-01-01,SAS,"Warriors look like a new team in thrilling, un...",knbr.com,kept:team_alias+context
2476,2020-01-01,GSW,Road warriors: Seahawks hoping away success co...,espn.com,kept:team_alias+context
2573,2020-01-01,MIA,Heat scorched by spin as Perth grab vital win,cricket.com.au,kept:team_alias+context


,news_date,team,title,source_name,relevance_reason
0,2020-01-01,None,December Spending /Saving: East Coast Saver,savingadvice.com,filtered:no_team_alias
1,2020-01-01,None,Ontario's 5-year electric scooter pilot launch...,torontosun.com,filtered:no_team_alias
2,2020-01-01,None,"Kulmeet Galhotra, longtime head of Cook County...",suntimes.com,filtered:no_team_alias
3,2020-01-01,None,Hong Kong's embattled leader praises police as...,qatar-tribune.com,filtered:no_team_alias
4,2020-01-01,None,SunLive - New Year's Honour &#x2013; 'I was re...,sunlive.co.nz,filtered:no_team_alias
5,2020-01-01,None,Low Levels of Environmental Pollutants May Slo...,medscape.com,filtered:no_team_alias
6,2020-01-01,None,Football: A window of opportunity for Arsenal ...,straitstimes.com,filtered:no_team_alias
7,2020-01-01,None,SunLive - SH2 traffic congestion expected toda...,sunlive.co.nz,filtered:no_team_alias
8,2020-01-01,None,Fansdoor Announces Algorithms for Social Media...,texasguardian.com,filtered:no_team_alias
9,2020-01-01,None,Matt Lauer's new girlfriend looks uncannily li...,dailymail.co.uk,filtered:no_team_alias


,news_date,team,title,source_name,relevance_decision,relevance_reason


## 10. Full Collection for 2020–2022

This cell runs full historical collection and may require substantial runtime.

In [ ]:
full_prefix = "nba_news_2020_2022"
full_raw, full_clean = process_month(FULL_START, FULL_END, full_prefix)
print(len(full_raw), len(full_clean))

GKG nba_news_2020_2022:   0%|          | 0/4384 [00:00<?, ?it/s]

4751738 11546


In [ ]:
raw_path_full = PATHS["raw"] / "nba_news_2020_2022_raw.parquet"
clean_path_full = PATHS["processed"] / "nba_news_2020_2022_clean.parquet"
if raw_path_full.exists() and clean_path_full.exists():
    raw_df = pd.read_parquet(raw_path_full)
    clean_df = pd.read_parquet(clean_path_full)
else:
    raw_df, clean_df = test_raw.copy(), test_clean.copy()

print("working raw:", raw_df.shape)
print("working clean:", clean_df.shape)

working raw: (4751738, 25)
working clean: (11546, 25)


## 11. Daily Team News Features

In [ ]:
# Агрегация признаков
def build_daily_features(clean_news: pd.DataFrame) -> pd.DataFrame:
    if clean_news.empty:
        return pd.DataFrame()
    d = clean_news.copy()
    d["news_date"] = pd.to_datetime(d["news_date"])
    out = d.groupby(["news_date", "team"], as_index=False).agg(
        news_count=("title", "size"),
        avg_gdelt_tone=("gdelt_tone", "mean"),
        min_gdelt_tone=("gdelt_tone", "min"),
        max_gdelt_tone=("gdelt_tone", "max"),
        avg_sentiment_score_kw=("sentiment_score_kw", "mean"),
        sum_positive_hits=("positive_hits", "sum"),
        sum_negative_hits=("negative_hits", "sum"),
        positive_news_count=("sentiment_label_kw", lambda s: (s == "positive").sum()),
        negative_news_count=("sentiment_label_kw", lambda s: (s == "negative").sum()),
        neutral_news_count=("sentiment_label_kw", lambda s: (s == "neutral").sum()),
        injury_news_count=("has_injury_kw", "sum"),
        trade_news_count=("has_trade_kw", "sum"),
        win_news_count=("has_win_kw", "sum"),
        loss_news_count=("has_loss_kw", "sum"),
        playoff_news_count=("has_playoff_kw", "sum"),
        roster_news_count=("has_roster_kw", "sum"),
        avg_title_len=("title_len", "mean"),
        avg_title_word_count=("title_word_count", "mean"),
    ).sort_values(["team", "news_date"])
    return out

daily = build_daily_features(clean_df)
display(daily.head(10))

,news_date,team,news_count,avg_gdelt_tone,min_gdelt_tone,max_gdelt_tone,avg_sentiment_score_kw,sum_positive_hits,sum_negative_hits,positive_news_count,negative_news_count,neutral_news_count,injury_news_count,trade_news_count,win_news_count,loss_news_count,playoff_news_count,roster_news_count,avg_title_len,avg_title_word_count
25,2020-01-04,ATL,3,-0.063511,-1.279318,1.880342,0.000000,0,0,0,0,3,0,3,0,0,0,0,71.666667,12.0
37,2020-01-05,ATL,1,-2.260870,-2.260870,-2.260870,0.000000,0,0,0,0,1,0,0,0,0,0,0,47.000000,8.0
50,2020-01-06,ATL,1,-1.426025,-1.426025,-1.426025,1.000000,1,0,1,0,0,0,0,1,0,0,0,53.000000,9.0
107,2020-01-11,ATL,2,-0.581710,-2.678571,1.515152,1.000000,2,0,2,0,0,0,0,2,0,0,0,61.500000,10.0
118,2020-01-13,ATL,1,-0.949796,-0.949796,-0.949796,1.000000,1,0,1,0,0,0,0,0,0,0,0,56.000000,10.0
142,2020-01-16,ATL,2,0.828125,-2.343750,4.000000,0.000000,0,0,0,0,2,0,1,0,0,0,0,86.000000,12.5
155,2020-01-17,ATL,1,1.171875,1.171875,1.171875,0.000000,0,0,0,0,1,0,1,0,0,0,0,66.000000,10.0
165,2020-01-18,ATL,3,-1.179518,-3.448276,0.996678,0.666667,2,0,2,0,1,0,1,2,0,0,0,59.666667,11.0
176,2020-01-19,ATL,1,2.857143,2.857143,2.857143,1.000000,1,0,1,0,0,0,0,1,0,0,0,64.000000,11.0
295,2020-01-29,ATL,4,0.811636,0.281690,1.137441,1.750000,7,0,4,0,0,0,0,4,0,0,0,66.500000,11.0


## 12. Lag and Rolling Features Without Data Leakage

In [ ]:
# Контроль утечки данных
def add_historical_features(daily_df: pd.DataFrame) -> pd.DataFrame:
    if daily_df.empty:
        return daily_df
    daily = daily_df.sort_values(["team", "news_date"]).copy()
    num_cols = [c for c in daily.columns if c not in ["news_date", "team"] and pd.api.types.is_numeric_dtype(daily[c])]
    for col in num_cols:
        daily[f"{col}_lag1"] = daily.groupby("team")[col].shift(1)
        daily[f"{col}_rollmean_3"] = daily.groupby("team")[col].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
        daily[f"{col}_rollmean_7"] = daily.groupby("team")[col].transform(lambda s: s.shift(1).rolling(7, min_periods=1).mean())
        daily[f"{col}_rollsum_3"] = daily.groupby("team")[col].transform(lambda s: s.shift(1).rolling(3, min_periods=1).sum())
        daily[f"{col}_rollsum_7"] = daily.groupby("team")[col].transform(lambda s: s.shift(1).rolling(7, min_periods=1).sum())
    return daily

daily_features = add_historical_features(daily)
display(daily_features.head(10))
print("feature columns:", len(daily_features.columns))

,news_date,team,news_count,avg_gdelt_tone,min_gdelt_tone,max_gdelt_tone,avg_sentiment_score_kw,sum_positive_hits,sum_negative_hits,positive_news_count,negative_news_count,neutral_news_count,injury_news_count,trade_news_count,win_news_count,loss_news_count,playoff_news_count,roster_news_count,avg_title_len,avg_title_word_count,news_count_lag1,news_count_rollmean_3,news_count_rollmean_7,news_count_rollsum_3,news_count_rollsum_7,avg_gdelt_tone_lag1,avg_gdelt_tone_rollmean_3,avg_gdelt_tone_rollmean_7,avg_gdelt_tone_rollsum_3,avg_gdelt_tone_rollsum_7,min_gdelt_tone_lag1,min_gdelt_tone_rollmean_3,min_gdelt_tone_rollmean_7,min_gdelt_tone_rollsum_3,min_gdelt_tone_rollsum_7,max_gdelt_tone_lag1,max_gdelt_tone_rollmean_3,max_gdelt_tone_rollmean_7,max_gdelt_tone_rollsum_3,max_gdelt_tone_rollsum_7,avg_sentiment_score_kw_lag1,avg_sentiment_score_kw_rollmean_3,avg_sentiment_score_kw_rollmean_7,avg_sentiment_score_kw_rollsum_3,avg_sentiment_score_kw_rollsum_7,sum_positive_hits_lag1,sum_positive_hits_rollmean_3,sum_positive_hits_rollmean_7,sum_positive_hits_rollsum_3,sum_positive_hits_rollsum_7,sum_negative_hits_lag1,sum_negative_hits_rollmean_3,sum_negative_hits_rollmean_7,sum_negative_hits_rollsum_3,sum_negative_hits_rollsum_7,positive_news_count_lag1,positive_news_count_rollmean_3,positive_news_count_rollmean_7,positive_news_count_rollsum_3,positive_news_count_rollsum_7,negative_news_count_lag1,negative_news_count_rollmean_3,negative_news_count_rollmean_7,negative_news_count_rollsum_3,negative_news_count_rollsum_7,neutral_news_count_lag1,neutral_news_count_rollmean_3,neutral_news_count_rollmean_7,neutral_news_count_rollsum_3,neutral_news_count_rollsum_7,injury_news_count_lag1,injury_news_count_rollmean_3,injury_news_count_rollmean_7,injury_news_count_rollsum_3,injury_news_count_rollsum_7,trade_news_count_lag1,trade_news_count_rollmean_3,trade_news_count_rollmean_7,trade_news_count_rollsum_3,trade_news_count_rollsum_7,win_news_count_lag1,win_news_count_rollmean_3,win_news_count_rollmean_7,win_news_count_rollsum_3,win_news_count_rollsum_7,loss_news_count_lag1,loss_news_count_rollmean_3,loss_news_count_rollmean_7,loss_news_count_rollsum_3,loss_news_count_rollsum_7,playoff_news_count_lag1,playoff_news_count_rollmean_3,playoff_news_count_rollmean_7,playoff_news_count_rollsum_3,playoff_news_count_rollsum_7,roster_news_count_lag1,roster_news_count_rollmean_3,roster_news_count_rollmean_7,roster_news_count_rollsum_3,roster_news_count_rollsum_7,avg_title_len_lag1,avg_title_len_rollmean_3,avg_title_len_rollmean_7,avg_title_len_rollsum_3,avg_title_len_rollsum_7,avg_title_word_count_lag1,avg_title_word_count_rollmean_3,avg_title_word_count_rollmean_7,avg_title_word_count_rollsum_3,avg_title_word_count_rollsum_7
25,2020-01-04,ATL,3,-0.063511,-1.279318,1.880342,0.000000,0,0,0,0,3,0,3,0,0,0,0,71.666667,12.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
37,2020-01-05,ATL,1,-2.260870,-2.260870,-2.260870,0.000000,0,0,0,0,1,0,0,0,0,0,0,47.000000,8.0,3.0,3.000000,3.000000,3.0,3.0,-0.063511,-0.063511,-0.063511,-0.063511,-0.063511,-1.279318,-1.279318,-1.279318,-1.279318,-1.279318,1.880342,1.880342,1.880342,1.880342,1.880342,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,3.000000,3.000000,3.0,3.0,0.0,0.0,0.0,0.0,0.0,3.0,3.000000,3.000000,3.0,3.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,71.666667,71.666667,71.666667,71.666667,71.666667,12.0,12.000000,12.000000,12.0,12.0
50,2020-01-06,ATL,1,-1.426025,-1.426025,-1.426025,1.000000,1,0,1,0,0,0,0,1,0,0,0,53.000000,9.0,1.0,2.000000,2.000000,4.0,4.0,-2.260870,-1.162

feature columns: 110


## 13. News Quality Report

In [ ]:
def build_quality_report(raw_df: pd.DataFrame, clean_df: pd.DataFrame) -> pd.DataFrame:
    sections = []
    overall = pd.DataFrame([{
        "report_type": "overall",
        "key": "all",
        "total_raw_news": int(len(raw_df)),
        "total_clean_news": int(len(clean_df)),
        "unique_teams": int(clean_df["team"].nunique()) if not clean_df.empty else 0,
        "date_min": str(pd.to_datetime(clean_df["news_date"]).min().date()) if not clean_df.empty else None,
        "date_max": str(pd.to_datetime(clean_df["news_date"]).max().date()) if not clean_df.empty else None,
        "share_filtered_out": float(1 - len(clean_df) / max(1, len(raw_df))),
    }])
    sections.append(overall)

    if not clean_df.empty:
        t = clean_df.copy()
        t["month"] = pd.to_datetime(t["news_date"]).dt.to_period("M").astype(str)
        by_month = t.groupby("month", as_index=False).agg(news_count=("title", "size"), unique_teams=("team", "nunique"), avg_gdelt_tone=("gdelt_tone", "mean"))
        by_month["report_type"] = "by_month"
        by_month.rename(columns={"month": "key"}, inplace=True)
        sections.append(by_month)

        by_team = t.groupby("team", as_index=False).agg(
            news_count=("title", "size"),
            avg_gdelt_tone=("gdelt_tone", "mean"),
            injury_news_count=("has_injury_kw", "sum"),
            trade_news_count=("has_trade_kw", "sum"),
            positive_news_count=("sentiment_label_kw", lambda s: (s == "positive").sum()),
            negative_news_count=("sentiment_label_kw", lambda s: (s == "negative").sum()),
            neutral_news_count=("sentiment_label_kw", lambda s: (s == "neutral").sum()),
        )
        by_team["report_type"] = "by_team"
        by_team.rename(columns={"team": "key"}, inplace=True)
        sections.append(by_team)

        by_source = t.groupby("source_name", as_index=False).agg(news_count=("title", "size")).sort_values("news_count", ascending=False).head(20)
        by_source["report_type"] = "by_source"
        by_source.rename(columns={"source_name": "key"}, inplace=True)
        sections.append(by_source)

        sent = t["sentiment_label_kw"].value_counts().rename_axis("key").reset_index(name="news_count")
        sent["report_type"] = "sentiment_distribution"
        sections.append(sent)

    return pd.concat(sections, ignore_index=True, sort=False)

quality_report = build_quality_report(raw_df, clean_df)
display(quality_report.head(30))

examples_kept = clean_df[["news_date", "team", "title", "source_name", "relevance_reason"]].head(200)
examples_filtered_out = raw_df[~raw_df["relevance_decision"]][["news_date", "team", "title", "source_name", "relevance_reason"]].head(200)
examples_ambiguous_aliases = raw_df[raw_df["title"].str.lower().str.contains(r"\\b(heat|magic|suns|kings|rockets|warriors)\\b", regex=True, na=False)][["news_date", "team", "title", "source_name", "relevance_decision", "relevance_reason"]].head(200)

,report_type,key,total_raw_news,total_clean_news,unique_teams,date_min,date_max,share_filtered_out,news_count,avg_gdelt_tone,injury_news_count,trade_news_count,positive_news_count,negative_news_count,neutral_news_count
0,overall,all,4751738.0,11546.0,30.0,2020-01-01,2022-12-31,0.99757,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,by_month,2020-01,NaN,NaN,30.0,NaN,NaN,NaN,578.0,-0.373270,NaN,NaN,NaN,NaN,NaN
2,by_month,2020-02,NaN,NaN,30.0,NaN,NaN,NaN,536.0,-0.554100,NaN,NaN,NaN,NaN,NaN
3,by_month,2020-03,NaN,NaN,29.0,NaN,NaN,NaN,293.0,-0.640733,NaN,NaN,NaN,NaN,NaN
4,by_month,2020-04,NaN,NaN,27.0,NaN,NaN,NaN,128.0,0.004966,NaN,NaN,NaN,NaN,NaN
5,by_month,2020-05,NaN,NaN,29.0,NaN,NaN,NaN,166.0,0.575950,NaN,NaN,NaN,NaN,NaN
6,by_month,2020-06,NaN,NaN,27.0,NaN,NaN,NaN,162.0,0.091970,NaN,NaN,NaN,NaN,NaN
7,by_month,2020-07,NaN,NaN,29.0,NaN,NaN,NaN,185.0,-0.004721,NaN,NaN,NaN,NaN,NaN
8,by_month,2020-08,NaN,NaN,28.0,NaN,NaN,NaN,521.0,-0.428032,NaN,NaN,NaN,NaN,NaN
9,by_month,2020-09,NaN,NaN,26.0,NaN,NaN,NaN,398.0,0.044950,NaN,NaN,NaN,NaN,NaN


## 14. Save Final Artifacts

In [ ]:
# Сохранение результатов
raw_df.to_parquet(PATHS["raw"] / "nba_news_raw.parquet", index=False)
clean_df.to_parquet(PATHS["processed"] / "nba_news_clean.parquet", index=False)
daily_features.to_csv(PATHS["processed"] / "nba_news_daily_features.csv", index=False)
quality_report.to_csv(PATHS["reports"] / "news_quality_report.csv", index=False)
examples_kept.to_csv(PATHS["artifacts"] / "examples_kept.csv", index=False)
examples_filtered_out.to_csv(PATHS["artifacts"] / "examples_filtered_out.csv", index=False)
examples_ambiguous_aliases.to_csv(PATHS["artifacts"] / "examples_ambiguous_aliases.csv", index=False)
print("Saved:")
print(PATHS["raw"] / "nba_news_raw.parquet")
print(PATHS["processed"] / "nba_news_clean.parquet")
print(PATHS["processed"] / "nba_news_daily_features.csv")
print(PATHS["reports"] / "news_quality_report.csv")
print(PATHS["artifacts"] / "examples_kept.csv")
print(PATHS["artifacts"] / "examples_filtered_out.csv")
print(PATHS["artifacts"] / "examples_ambiguous_aliases.csv")

Saved:
/content/data/news/raw/nba_news_raw.parquet
/content/data/news/processed/nba_news_clean.parquet
/content/data/news/processed/nba_news_daily_features.csv
/content/data/news/reports/news_quality_report.csv
/content/artifacts/news/examples_kept.csv
/content/artifacts/news/examples_filtered_out.csv
/content/artifacts/news/examples_ambiguous_aliases.csv


## 15. Create docs/news_dataset.md

In [ ]:
doc_text = """# NBA News Dataset

## 1. Назначение датасета
Датасет формирует внешние информационные признаки команд NBA для прогнозирования результата матча как дополнительный источник данных.

## 2. Источник данных
GDELT GKG 2.1 — открытый исторический новостной источник для воспроизводимых исследований.

## 3. Период сбора
2020-01-01 — 2022-12-31.

## 4. Единица наблюдения
- Raw/Clean: одна строка = одна новость, связанная с одной NBA-командой.
- Daily features: одна строка = одна команда в один календарный день.

## 5. Логика фильтрации
- алиасы команд;
- NBA-контекст;
- whitelist спортивных источников;
- exclusion context;
- контроль ложных совпадений коротких алиасов.

## 6. Список признаков
- идентификаторы;
- sentiment;
- keyword flags;
- daily aggregation;
- lag/rolling features.

## 7. Контроль data leakage
Текущий день не используется напрямую для предматчевого прогноза. Применяются lag1 и rolling через shift(1) по каждой команде.

## 8. Ограничения
- keyword sentiment не является entity-level sentiment;
- возможны ложные совпадения;
- покрытие GDELT неравномерно;
- новости по командам несбалансированы;
- признаки являются дополнительными, а не основными.

## 9. Использование на следующих этапах
Файл nba_news_daily_features.csv объединяется с match-level dataset по team и game_date.

## 10. Практическая ценность
Новости отражают травмы, обмены, кризисы, успешные серии, playoff-контекст и общий информационный фон команды.
"""

doc_path = PATHS["docs"] / "news_dataset.md"
doc_path.write_text(doc_text, encoding="utf-8")
print(doc_path)

/content/docs/news_dataset.md


## 16. Final Research Summary

Collected: raw and clean NBA news datasets from GDELT GKG 2.1 for the target period.

Generated features: team-day aggregates, keyword sentiment, topic flags, lag and rolling statistics.

Data leakage control: all historical features use shift(1), excluding current-day news from pregame signal.

Limitations: noisy headlines, ambiguous aliases, source coverage imbalance, and simplified keyword sentiment.

Next step: merge `nba_news_daily_features.csv` with match-level data by `team` and `game_date`.